In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q19/q19_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# Define container and ship mode lists using string literals
SM_SMALL = ["SM BOX", "SM CASE", "SM PACK", "SM PKG"]
MED_CONTAINER = ["MED BAG", "MED BOX", "MED PACK", "MED PKG"]
LG_CONTAINER = ["LG BOX", "LG CASE", "LG PACK", "LG PKG"]
SHIPMODES = ["AIR", "AIRREG"]

# Filter lineitem with vectorized between and isin
li_qty = (
    lineitem.L_QUANTITY.between(4, 14)
    | lineitem.L_QUANTITY.between(15, 25)
    | lineitem.L_QUANTITY.between(26, 36)
)
li_ship = (
    (lineitem.L_SHIPINSTRUCT == "DELIVER IN PERSON")
    & lineitem.L_SHIPMODE.isin(SHIPMODES)
)
flineitem = lineitem[li_qty & li_ship]

# Filter part using isin and between
small_cond = (
    (part.P_BRAND == "Brand#31")
    & part.P_CONTAINER.isin(SM_SMALL)
    & part.P_SIZE.between(1, 5)
)
med_cond = (
    (part.P_BRAND == "Brand#43")
    & part.P_CONTAINER.isin(MED_CONTAINER)
    & part.P_SIZE.between(1, 10)
)
lg_cond = (
    (part.P_BRAND == "Brand#43")
    & part.P_CONTAINER.isin(LG_CONTAINER)
    & part.P_SIZE.between(1, 15)
)
fpart = part[small_cond | med_cond | lg_cond]

# Join and apply final filtering
jn = flineitem.merge(fpart, left_on="L_PARTKEY", right_on="P_PARTKEY")

cond1 = (
    (jn.P_BRAND == "Brand#31")
    & jn.P_CONTAINER.isin(SM_SMALL)
    & jn.L_QUANTITY.between(4, 14)
    & (jn.P_SIZE <= 5)
)
cond2 = (
    (jn.P_BRAND == "Brand#43")
    & jn.P_CONTAINER.isin(MED_CONTAINER)
    & jn.L_QUANTITY.between(15, 25)
    & (jn.P_SIZE <= 10)
)
cond3 = (
    (jn.P_BRAND == "Brand#43")
    & jn.P_CONTAINER.isin(LG_CONTAINER)
    & jn.L_QUANTITY.between(26, 36)
    & (jn.P_SIZE <= 15)
)
j = jn[cond1 | cond2 | cond3]

# Compute total
# Note: using cudf, this will execute on GPU
total = (j.L_EXTENDEDPRICE * (1.0 - j.L_DISCOUNT)).sum()